# 01. Data Exploration - BioSNAP Polypharmacy

**Đồ án:** GNN Protein Function Prediction  
**Môn học:** IS353 - Mạng Xã Hội

## Mục tiêu
1. Download và khám phá dataset BioSNAP-Polypharmacy
2. Mô tả định dạng file và cấu trúc dữ liệu
3. Trực quan hóa phân bổ relation types
4. Vẽ subgraph với màu sắc theo relation

## 1. Setup & Installation

In [ ]:
# Install dependencies (chạy trên Colab)
!pip install torch torch-geometric torch-scatter torch-sparse -q
!pip install pandas numpy matplotlib seaborn networkx -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Matplotlib settings
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

## 2. Download Dataset

**BioSNAP-Polypharmacy Dataset**
- Source: Stanford BioSNAP
- URL: http://snap.stanford.edu/biodata/
- Description: Drug-drug polypharmacy side effects network

In [ ]:
import urllib.request
import gzip
import os

# Create data directory
os.makedirs('data', exist_ok=True)

# Dataset URL
POLYPHARMACY_URL = 'http://snap.stanford.edu/biodata/datasets/10017/files/ChChSe-Decagon_polypharmacy.csv.gz'

def download_dataset(url, output_path):
    """Download and extract gzip file"""
    gz_path = output_path + '.gz'
    
    if os.path.exists(output_path):
        print(f"File already exists: {output_path}")
        return
    
    print(f"Downloading from {url}...")
    urllib.request.urlretrieve(url, gz_path)
    
    print("Extracting...")
    with gzip.open(gz_path, 'rb') as f_in:
        with open(output_path, 'wb') as f_out:
            f_out.write(f_in.read())
    
    os.remove(gz_path)
    print(f"Done! Saved to {output_path}")

# Download
download_dataset(POLYPHARMACY_URL, 'data/polypharmacy.csv')

## 3. Mô tả Dữ liệu

### 3.1 Định dạng file

In [ ]:
# Load dataset
df = pd.read_csv('data/polypharmacy.csv')

# Rename columns for clarity
df.columns = ['Drug1', 'Drug2', 'SideEffect']

print("="*60)
print("ĐỊNH DẠNG FILE")
print("="*60)
print(f"File type: CSV (Comma Separated Values)")
print(f"Columns: {list(df.columns)}")
print(f"\nMô tả columns:")
print(f"  - Drug1: STITCH ID của thuốc thứ nhất")
print(f"  - Drug2: STITCH ID của thuốc thứ hai")
print(f"  - SideEffect: Tên tác dụng phụ khi dùng 2 thuốc cùng lúc")
print(f"\nFirst 5 rows:")
df.head()

In [ ]:
print("="*60)
print("THỐNG KÊ TỔNG QUAN")
print("="*60)

num_edges = len(df)
num_drugs = len(set(df['Drug1'].unique()) | set(df['Drug2'].unique()))
num_relations = df['SideEffect'].nunique()

print(f"Số lượng edges (interactions): {num_edges:,}")
print(f"Số lượng nodes (drugs): {num_drugs:,}")
print(f"Số lượng relation types (side effects): {num_relations}")
print(f"\n→ Dataset có HƠN 200 loại quan hệ (tác dụng phụ)")

### 3.2 Mục đích dữ liệu

**Polypharmacy** = Sử dụng nhiều loại thuốc cùng lúc

**Vấn đề thực tế:**
- Khi bệnh nhân dùng nhiều thuốc, có thể xảy ra tác dụng phụ không mong muốn
- Ví dụ: Thuốc A + Thuốc B → Gây buồn nôn (side effect)

**Mục đích:**
- Dự đoán tác dụng phụ khi kết hợp 2 loại thuốc
- Giúp bác sĩ kê đơn an toàn hơn
- Multi-relational link prediction: Dự đoán loại liên kết cụ thể giữa 2 nodes

### 3.3 Chuyển thành format (Subject, Relation, Object)

In [ ]:
print("="*60)
print("CHUYỂN ĐỔI THÀNH TRIPLES (S, R, O)")
print("="*60)

# Already in triple format!
triples_df = df.copy()
triples_df.columns = ['Subject', 'Object', 'Relation']

print("Format: (Subject, Relation, Object)")
print("  - Subject: Drug 1")
print("  - Relation: Side Effect type")
print("  - Object: Drug 2")
print("\nVí dụ:")
for i, row in triples_df.head(3).iterrows():
    print(f"  ({row['Subject']}, '{row['Relation']}', {row['Object']})")

## 4. Trực quan hóa

### 4.1 Relation Distribution (Phân bổ loại quan hệ)

In [ ]:
# Count relation types
relation_counts = df['SideEffect'].value_counts()

print(f"Top 20 side effects (relation types):")
print(relation_counts.head(20))

In [ ]:
# Plot: Top 20 relations
fig, ax = plt.subplots(figsize=(14, 8))

top_20 = relation_counts.head(20)
colors = plt.cm.viridis(np.linspace(0, 0.8, len(top_20)))

bars = ax.barh(range(len(top_20)), top_20.values, color=colors)
ax.set_yticks(range(len(top_20)))
ax.set_yticklabels(top_20.index)
ax.invert_yaxis()
ax.set_xlabel('Number of Drug Pairs', fontsize=12)
ax.set_title('Top 20 Polypharmacy Side Effects (Relation Distribution)', fontsize=14, fontweight='bold')

# Add value labels
for i, (bar, val) in enumerate(zip(bars, top_20.values)):
    ax.text(val + 500, i, f'{val:,}', va='center', fontsize=10)

plt.tight_layout()
plt.savefig('figures/relation_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✅ Saved: figures/relation_distribution.png")

In [ ]:
# Plot: Distribution histogram
fig, ax = plt.subplots(figsize=(12, 6))

ax.hist(relation_counts.values, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(relation_counts.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {relation_counts.mean():.0f}')
ax.axvline(relation_counts.median(), color='orange', linestyle='--', linewidth=2, label=f'Median: {relation_counts.median():.0f}')

ax.set_xlabel('Number of Drug Pairs per Side Effect', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_title('Distribution of Side Effect Frequencies', fontsize=14, fontweight='bold')
ax.legend()

plt.tight_layout()
plt.savefig('figures/relation_frequency_hist.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.2 Node Degree Distribution

In [ ]:
# Calculate node degrees
all_drugs = list(df['Drug1']) + list(df['Drug2'])
degree_counts = Counter(all_drugs)
degrees = list(degree_counts.values())

print(f"Degree Statistics:")
print(f"  Min degree: {min(degrees)}")
print(f"  Max degree: {max(degrees)}")
print(f"  Mean degree: {np.mean(degrees):.2f}")
print(f"  Median degree: {np.median(degrees):.2f}")

In [ ]:
# Plot degree distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Linear scale
axes[0].hist(degrees, bins=50, color='coral', edgecolor='white', alpha=0.8)
axes[0].set_xlabel('Degree', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('Node Degree Distribution (Linear)', fontsize=12, fontweight='bold')

# Log scale
axes[1].hist(degrees, bins=50, color='coral', edgecolor='white', alpha=0.8)
axes[1].set_yscale('log')
axes[1].set_xlabel('Degree', fontsize=12)
axes[1].set_ylabel('Frequency (log)', fontsize=12)
axes[1].set_title('Node Degree Distribution (Log Scale)', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('figures/degree_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✅ Saved: figures/degree_distribution.png")

### 4.3 Subgraph Visualization (Colored by Relation)

In [ ]:
# Select top 5 relations and sample drugs
top_5_relations = relation_counts.head(5).index.tolist()
print(f"Top 5 relations for visualization:")
for i, rel in enumerate(top_5_relations, 1):
    print(f"  {i}. {rel}")

In [ ]:
# Filter data for top 5 relations
df_subset = df[df['SideEffect'].isin(top_5_relations)]

# Get most connected drugs
drug_counts = Counter(list(df_subset['Drug1']) + list(df_subset['Drug2']))
top_drugs = [drug for drug, _ in drug_counts.most_common(15)]

# Filter to create small subgraph
df_subgraph = df_subset[
    (df_subset['Drug1'].isin(top_drugs)) & 
    (df_subset['Drug2'].isin(top_drugs))
].head(100)

print(f"Subgraph stats:")
print(f"  Edges: {len(df_subgraph)}")
print(f"  Unique drugs: {len(set(df_subgraph['Drug1']) | set(df_subgraph['Drug2']))}")

In [ ]:
# Create MultiGraph (allows multiple edges between nodes)
G = nx.MultiGraph()

# Color map for relations
relation_colors = {
    top_5_relations[0]: '#e74c3c',  # Red
    top_5_relations[1]: '#3498db',  # Blue
    top_5_relations[2]: '#2ecc71',  # Green
    top_5_relations[3]: '#9b59b6',  # Purple
    top_5_relations[4]: '#f39c12',  # Orange
}

# Add edges
edge_colors = []
for _, row in df_subgraph.iterrows():
    G.add_edge(row['Drug1'], row['Drug2'], relation=row['SideEffect'])
    edge_colors.append(relation_colors.get(row['SideEffect'], 'gray'))

# Plot
fig, ax = plt.subplots(figsize=(14, 10))

pos = nx.spring_layout(G, k=2, iterations=50, seed=42)

# Draw nodes
nx.draw_networkx_nodes(G, pos, node_color='lightblue', node_size=800, 
                       edgecolors='darkblue', linewidths=2, ax=ax)

# Draw edges with colors
nx.draw_networkx_edges(G, pos, edge_color=edge_colors, width=2, alpha=0.7, ax=ax)

# Draw labels (shortened drug IDs)
labels = {node: node[-4:] for node in G.nodes()}
nx.draw_networkx_labels(G, pos, labels, font_size=8, font_weight='bold', ax=ax)

# Legend
legend_elements = [plt.Line2D([0], [0], color=color, linewidth=3, label=rel[:30]+'...' if len(rel)>30 else rel) 
                   for rel, color in relation_colors.items()]
ax.legend(handles=legend_elements, loc='upper left', fontsize=9, title='Side Effects')

ax.set_title('Drug-Drug Interaction Subgraph\n(Colored by Side Effect Type)', 
             fontsize=14, fontweight='bold')
ax.axis('off')

plt.tight_layout()
plt.savefig('figures/subgraph_colored.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✅ Saved: figures/subgraph_colored.png")

## 5. Graph Characteristics (Đặc trưng đồ thị)

In [ ]:
# Create simple graph (ignore edge types for basic stats)
G_simple = nx.Graph()
for _, row in df.iterrows():
    G_simple.add_edge(row['Drug1'], row['Drug2'])

print("="*60)
print("ĐẶC TRƯNG GRAPH")
print("="*60)

print(f"\n1. Basic Statistics:")
print(f"   Nodes: {G_simple.number_of_nodes()}")
print(f"   Edges: {G_simple.number_of_edges()}")
print(f"   Density: {nx.density(G_simple):.6f}")

print(f"\n2. Connected Components:")
components = list(nx.connected_components(G_simple))
print(f"   Number of components: {len(components)}")
print(f"   Largest component size: {len(max(components, key=len))}")

print(f"\n3. Degree Statistics:")
degrees = [d for n, d in G_simple.degree()]
print(f"   Average degree: {np.mean(degrees):.2f}")
print(f"   Max degree: {max(degrees)}")
print(f"   Min degree: {min(degrees)}")

In [ ]:
# Sample for expensive computations
print(f"\n4. Clustering Coefficient (sampled):")
# Use sampling for large graphs
sample_nodes = list(G_simple.nodes())[:100]
clustering_sample = nx.clustering(G_simple, sample_nodes)
avg_clustering = np.mean(list(clustering_sample.values()))
print(f"   Average clustering (sample): {avg_clustering:.4f}")

print(f"\n5. Top 10 Central Nodes (by degree):")
degree_centrality = nx.degree_centrality(G_simple)
top_central = sorted(degree_centrality.items(), key=lambda x: x[1], reverse=True)[:10]
for i, (node, cent) in enumerate(top_central, 1):
    print(f"   {i}. {node}: {cent:.4f}")

## 6. Summary

### Dataset Statistics

In [ ]:
print("="*60)
print("TÓM TẮT DATASET")
print("="*60)
print(f"""
📊 BioSNAP-Polypharmacy Dataset

1. ĐỊNH DẠNG:
   - File: CSV
   - Columns: Drug1, Drug2, SideEffect
   - Format: (Subject, Relation, Object) triples

2. THỐNG KÊ:
   - Số edges: {num_edges:,}
   - Số nodes (drugs): {num_drugs:,}
   - Số relation types: {num_relations} (>200 loại tác dụng phụ)

3. MỤC ĐÍCH:
   - Dự đoán tác dụng phụ khi dùng nhiều thuốc cùng lúc
   - Multi-relational link prediction
   - Hỗ trợ kê đơn thuốc an toàn

4. ĐẶC ĐIỂM:
   - Graph density: {nx.density(G_simple):.6f}
   - Average degree: {np.mean(degrees):.2f}
   - Nhiều loại quan hệ → Cần Multi-relational GNN (R-GCN)
""")

In [ ]:
# Save processed data for next notebooks
df.to_csv('data/polypharmacy_processed.csv', index=False)
print("✅ Saved processed data: data/polypharmacy_processed.csv")

# Save statistics
stats = {
    'num_edges': num_edges,
    'num_nodes': num_drugs,
    'num_relations': num_relations,
    'avg_degree': np.mean(degrees),
    'density': nx.density(G_simple),
}

import json
with open('data/dataset_stats.json', 'w') as f:
    json.dump(stats, f, indent=2)
print("✅ Saved statistics: data/dataset_stats.json")

---

## ✅ Checklist Phase 1

- [x] Download BioSNAP dataset
- [x] Mô tả định dạng file (CSV, 3 columns)
- [x] Mô tả dữ liệu (>200 relation types)
- [x] Chuyển thành (S, R, O) triples
- [x] Trực quan: Relation distribution
- [x] Trực quan: Subgraph với màu theo relation
- [x] Tính graph characteristics

**Next:** Phase 2 - Model Implementation (GAE + VGAE)